In [23]:
import requests
import os
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, set_tracing_export_api_key

load_dotenv(override=True)

# LLM calls go through OpenRouter; traces still export to OpenAI's dashboard.
set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))

openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model = OpenAIChatCompletionsModel(
    model="openai/gpt-4o-mini",  # OpenRouter model slugs look like provider/model
    openai_client=openrouter_client,
)


In [10]:
reader = PdfReader("osama_linkedin.pdf")

linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

TWIN_SYSTEM_PROMPT = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.

Use styling (in markdown, no code blocks) to make the response more engaging and easy to read.
""".strip()


In [11]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [17]:
def push(text):
    requests.post(
        pushover_url,
        data={
            "token": pushover_token,
            "user": pushover_user,
            "message": text,
        },
    )

In [26]:
@function_tool
def record_user_details(email : str, name : str="Name not provided", notes : str="not provided"):
    """ Use this tool to record that a user is interested in being in touch and provided an email address """
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [27]:
record_user_details.params_json_schema

{'properties': {'email': {'title': 'Email', 'type': 'string'},
  'name': {'default': 'Name not provided', 'title': 'Name', 'type': 'string'},
  'notes': {'default': 'not provided', 'title': 'Notes', 'type': 'string'}},
 'required': ['email', 'name', 'notes'],
 'title': 'record_user_details_args',
 'type': 'object',
 'additionalProperties': False}

In [28]:
@function_tool
def record_unknown_question(question : str):
    """ Use this tool to record that a user asked a question that I couldn't answer """
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [29]:
agent = Agent(
    name="Digital Twin",
    instructions=TWIN_SYSTEM_PROMPT,
    model=model,
    tools=[record_user_details, record_unknown_question],
)


In [ ]:
with trace("First test of the digital twin"):
    result = await Runner.run(agent, "Summarize the person's LinkedIn profile")
print(result.final_output)